# Phase 1: Batch NTFS Artifact Parser

## Overview
This notebook consolidates MFT, LogFile, and UsnJrnl parsers into a unified batch processing pipeline.

### Purpose
Process multiple datasets incrementally:
- Parse **$MFT** → Extract file metadata and timestamps
- Parse **$LogFile** → Extract transaction records and timestamp changes
- Parse **$UsnJrnl** → Extract change journal events

### Dataset Structure

- **PE Cases**: 01-PE → 12-PE (12 datasets)
- **APT Cases**: 01-APT17, 02-APT19, 03-APT21, 04-APT28, 05-APT29, 06-APT30, 07-APT37, 08-APT38, 09-APT40, 10-DarkHotel663, 11-DarkHotelbbd, 12-Kimsuky, 13-Winnti731, 14-Winnti43b (14 datasets)
- **LoneWolf** (1 dataset)



In [21]:
# [Cell 1] Install and Import Dependencies
import subprocess
import sys

def install_package(package):
    """Install a package using pip."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Install dfir_ntfs from GitHub
try:
    import dfir_ntfs
    print(f"dfir_ntfs version: {dfir_ntfs.__version__}")
except ImportError:
    print("Installing dfir_ntfs...")
    install_package("git+https://github.com/bamonskiy-kaban/dfir_ntfs.git")
    import dfir_ntfs
    print(f"dfir_ntfs installed successfully. Version: {dfir_ntfs.__version__}")

# Import other dependencies
import os
import struct
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

# dfir_ntfs modules
from dfir_ntfs import MFT
from dfir_ntfs.USN import ChangeJournalParser, ResolveReasonCodes
from dfir_ntfs.LogFile import LogFileParser

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("All libraries imported successfully.")


dfir_ntfs version: 1.1.19
All libraries imported successfully.


In [22]:
# [Cell 2] Define Base Paths and Configuration

# Base project directory
PROJECT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis")

# Input base directories
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PE_DIR = RAW_DATA_DIR / "PE"
APT_DIR = RAW_DATA_DIR / "APT"
LONEWOLF_DIR = RAW_DATA_DIR / "LoneWolf"

# Output base directory
OUTPUT_BASE_DIR = PROJECT_DIR / "data" / "Phase 1: Raw Data Parsing"

# Dataset lists
PE_DATASETS = [f"{i:02d}-PE" for i in range(1, 13)]  # 01-PE to 12-PE

APT_DATASETS = [
    "01-APT17",
    "02-APT19",
    "03-APT21",
    "04-APT28",
    "05-APT29",
    "06-APT30",
    "07-APT37",
    "08-APT38",
    "09-APT40",
    "10-DarkHotel663",
    "11-DarkHotelbbd",
    "12-Kimsuky",
    "13-Winnti731",
    "14-Winnti43b"
]

LONEWOLF_DATASETS = ["LoneWolf"]

print(f"Project directory: {PROJECT_DIR}")
print(f"\nPE datasets: {len(PE_DATASETS)}")
print(f"APT datasets: {len(APT_DATASETS)}")
print(f"LoneWolf datasets: {len(LONEWOLF_DATASETS)}")
print(f"\nTotal datasets: {len(PE_DATASETS) + len(APT_DATASETS) + len(LONEWOLF_DATASETS)}")


Project directory: /Users/soni/Github/Digital-Detectives_Thesis

PE datasets: 12
APT datasets: 14
LoneWolf datasets: 1

Total datasets: 27


## MFT Parser Functions

The following cells define functions for parsing MFT files.

In [23]:
# [Cell 3] MFT Parser - Helper Functions

def mft_format_timestamp(dt_obj):
    """Format datetime object to string with microsecond precision."""
    if dt_obj is None:
        return None
    try:
        return dt_obj.strftime("%Y-%m-%d %H:%M:%S.%f")
    except (ValueError, AttributeError):
        return None


def mft_extract_si_timestamps(file_record):
    """Extract $STANDARD_INFORMATION timestamps from a file record."""
    si_timestamps = {
        "SI_C": None,
        "SI_M": None,
        "SI_E": None,
        "SI_A": None
    }
    
    try:
        for attribute in file_record.attributes():
            if attribute.type_code == 0x10:  # $STANDARD_INFORMATION
                try:
                    si = attribute.value_decoded()
                    si_timestamps["SI_C"] = mft_format_timestamp(si.get_ctime())
                    si_timestamps["SI_M"] = mft_format_timestamp(si.get_mtime())
                    si_timestamps["SI_E"] = mft_format_timestamp(si.get_etime())
                    si_timestamps["SI_A"] = mft_format_timestamp(si.get_atime())
                    break
                except Exception:
                    pass
    except Exception:
        pass
    
    return si_timestamps


def mft_extract_fn_info(file_record, parser):
    """Extract $FILE_NAME attribute information including timestamps."""
    fn_info = {
        "FileName": None,
        "ParentFRN": None,
        "FN_C": None,
        "FN_M": None,
        "FN_E": None,
        "FN_A": None
    }
    
    try:
        for attribute in file_record.attributes():
            if attribute.type_code == 0x30:  # $FILE_NAME
                try:
                    fn = attribute.value_decoded()
                    file_name = fn.get_file_name()
                    namespace = fn.get_flags()
                    
                    if fn_info["FileName"] is None or namespace in (1, 3):
                        fn_info["FileName"] = file_name
                        fn_info["ParentFRN"] = fn.get_parent_directory() & 0xFFFFFFFFFFFF
                        fn_info["FN_C"] = mft_format_timestamp(fn.get_ctime())
                        fn_info["FN_M"] = mft_format_timestamp(fn.get_mtime())
                        fn_info["FN_E"] = mft_format_timestamp(fn.get_etime())
                        fn_info["FN_A"] = mft_format_timestamp(fn.get_atime())
                        
                        if namespace in (1, 3):
                            break
                except Exception:
                    pass
    except Exception:
        pass
    
    return fn_info


def mft_build_file_path(file_record, parser, path_cache):
    """Build the full file path for a given file record."""
    try:
        paths = parser.build_full_paths(file_record)
        if paths:
            return paths[0][0] if isinstance(paths[0], tuple) else paths[0]
    except Exception:
        pass
    return None

print("MFT helper functions defined.")


MFT helper functions defined.


In [24]:
# [Cell 4] MFT Parser - Main Function

def parse_mft(mft_path, progress_interval=50000):
    """
    Parse the MFT file and extract all relevant metadata.
    
    Args:
        mft_path: Path to the $MFT file
        progress_interval: Print progress every N records
        
    Returns:
        pandas.DataFrame: Parsed MFT data
    """
    records = []
    path_cache = {}
    
    print(f"  Opening MFT file: {mft_path.name}")
    
    with open(mft_path, "rb") as mft_file:
        parser = MFT.MasterFileTableParser(mft_file)
        
        record_count = 0
        error_count = 0
        
        for file_record in parser.file_records():
            try:
                record_count += 1
                
                if record_count % progress_interval == 0:
                    print(f"    Processed {record_count:,} records...")
                
                entry_number = file_record.get_master_file_table_number()
                is_active = file_record.is_in_use()
                lsn = file_record.get_logfile_sequence_number()
                
                si_ts = mft_extract_si_timestamps(file_record)
                fn_info = mft_extract_fn_info(file_record, parser)
                file_path = mft_build_file_path(file_record, parser, path_cache)
                
                record = {
                    "EntryNumber": entry_number,
                    "FileName": fn_info["FileName"],
                    "FilePath": file_path,
                    "IsActive": is_active,
                    "LSN": lsn,
                    "ParentFRN": fn_info["ParentFRN"],
                    "$SI-C": si_ts["SI_C"],
                    "$SI-M": si_ts["SI_M"],
                    "$SI-E": si_ts["SI_E"],
                    "$SI-A": si_ts["SI_A"],
                    "$FN-C": fn_info["FN_C"],
                    "$FN-M": fn_info["FN_M"],
                    "$FN-E": fn_info["FN_E"],
                    "$FN-A": fn_info["FN_A"]
                }
                
                records.append(record)
                
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f"    Warning: Error parsing record {record_count}: {str(e)[:80]}")
    
    print(f"    Total: {record_count:,} records, Errors: {error_count:,}, Parsed: {len(records):,}")
    
    return pd.DataFrame(records)

print("MFT parser function defined.")


MFT parser function defined.


## UsnJrnl Parser Functions

The following cells define functions for parsing UsnJrnl files.

In [25]:
# [Cell 5] UsnJrnl Parser - Helper Functions and Constants

# Reason flags
USN_REASON_FLAGS = {
    0x00000001: "DATA_OVERWRITE",
    0x00000002: "DATA_EXTEND",
    0x00000004: "DATA_TRUNCATION",
    0x00000010: "NAMED_DATA_OVERWRITE",
    0x00000020: "NAMED_DATA_EXTEND",
    0x00000040: "NAMED_DATA_TRUNCATION",
    0x00000100: "FILE_CREATE",
    0x00000200: "FILE_DELETE",
    0x00001000: "EA_CHANGE",
    0x00002000: "SECURITY_CHANGE",
    0x00004000: "RENAME_OLD_NAME",
    0x00008000: "RENAME_NEW_NAME",
    0x00010000: "INDEXABLE_CHANGE",
    0x00020000: "BASIC_INFO_CHANGE",
    0x00040000: "HARD_LINK_CHANGE",
    0x00080000: "COMPRESSION_CHANGE",
    0x00100000: "ENCRYPTION_CHANGE",
    0x00200000: "OBJECT_ID_CHANGE",
    0x00400000: "REPARSE_POINT_CHANGE",
    0x00800000: "STREAM_CHANGE",
    0x80000000: "CLOSE"
}


def usn_format_timestamp(dt_obj):
    """Format datetime object to string with microsecond precision."""
    if dt_obj is None:
        return None
    try:
        return dt_obj.strftime("%Y-%m-%d %H:%M:%S.%f")
    except (ValueError, AttributeError):
        return None


def usn_parse_reason_flags(reason_code):
    """Parse reason code integer into list of flag names."""
    if reason_code == 0:
        return "NONE"
    
    flags = []
    for mask, name in USN_REASON_FLAGS.items():
        if reason_code & mask:
            flags.append(name)
    
    return " | ".join(flags) if flags else f"UNKNOWN_0x{reason_code:08X}"


def usn_check_basic_detection_pattern(reason_code):
    """Check if the reason code contains BASIC_INFO_CHANGE flag."""
    return bool(reason_code & 0x00020000)


def usn_check_close_pattern(reason_code):
    """Check if the reason code contains CLOSE flag."""
    return bool(reason_code & 0x80000000)


def usn_check_file_create_pattern(reason_code):
    """Check if the reason code contains FILE_CREATE flag."""
    return bool(reason_code & 0x00000100)

print("UsnJrnl helper functions defined.")


UsnJrnl helper functions defined.


In [26]:
# [Cell 6] UsnJrnl Parser - Main Function

def parse_usnjrnl(usnjrnl_path, progress_interval=10000):
    """
    Parse the UsnJrnl file and extract all relevant metadata.
    
    Args:
        usnjrnl_path: Path to the $UsnJrnl:$J file
        progress_interval: Print progress every N records
        
    Returns:
        pandas.DataFrame: Parsed UsnJrnl data
    """
    records = []
    
    print(f"  Opening UsnJrnl file: {usnjrnl_path.name}")
    
    with open(usnjrnl_path, "rb") as usn_file:
        parser = ChangeJournalParser(usn_file)
        
        record_count = 0
        error_count = 0
        
        for usn_record in parser.usn_records():
            try:
                record_count += 1
                
                if record_count % progress_interval == 0:
                    print(f"    Processed {record_count:,} records...")
                
                usn = usn_record.get_usn()
                frn = usn_record.get_file_reference_number() & 0xFFFFFFFFFFFF
                parent_frn = usn_record.get_parent_file_reference_number() & 0xFFFFFFFFFFFF
                timestamp = usn_format_timestamp(usn_record.get_timestamp())
                filename = usn_record.get_file_name()
                reason_code = usn_record.get_reason()
                reason_flags = usn_parse_reason_flags(reason_code)
                source_info = usn_record.get_source_info()
                
                has_basic_info_change = usn_check_basic_detection_pattern(reason_code)
                has_close = usn_check_close_pattern(reason_code)
                has_file_create = usn_check_file_create_pattern(reason_code)
                
                record = {
                    "USN": usn,
                    "FRN": frn,
                    "ParentFRN": parent_frn,
                    "Timestamp": timestamp,
                    "FileName": filename,
                    "ReasonCode": reason_code,
                    "ReasonFlags": reason_flags,
                    "SourceInfo": source_info,
                    "HasBasicInfoChange": has_basic_info_change,
                    "HasClose": has_close,
                    "HasFileCreate": has_file_create
                }
                
                records.append(record)
                
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f"    Warning: Error parsing record {record_count}: {str(e)[:80]}")
    
    print(f"    Total: {record_count:,} records, Errors: {error_count:,}, Parsed: {len(records):,}")
    
    return pd.DataFrame(records)

print("UsnJrnl parser function defined.")


UsnJrnl parser function defined.


## LogFile Parser Functions

The following cells define functions for parsing LogFile.

In [27]:
# [Cell 7] LogFile Parser - Helper Functions and Constants

# NTFS Operation Codes
LOGFILE_OPERATION_CODES = {
    0x00: "Noop",
    0x01: "CompensationLogRecord",
    0x02: "InitializeFileRecordSegment",
    0x03: "DeallocateFileRecordSegment",
    0x04: "WriteEndOfFileRecordSegment",
    0x05: "CreateAttribute",
    0x06: "DeleteAttribute",
    0x07: "UpdateResidentValue",
    0x08: "UpdateNonresidentValue",
    0x09: "UpdateMappingPairs",
    0x0A: "DeleteDirtyClusters",
    0x0B: "SetNewAttributeSizes",
    0x0C: "AddIndexEntryRoot",
    0x0D: "DeleteIndexEntryRoot",
    0x0E: "AddIndexEntryAllocation",
    0x0F: "DeleteIndexEntryAllocation",
    0x10: "WriteEndOfIndexBuffer",
    0x11: "SetIndexEntryVcnRoot",
    0x12: "SetIndexEntryVcnAllocation",
    0x13: "UpdateFileNameRoot",
    0x14: "UpdateFileNameAllocation",
    0x15: "SetBitsInNonresidentBitMap",
    0x16: "ClearBitsInNonresidentBitMap",
    0x17: "HotFix",
    0x18: "EndTopLevelAction",
    0x19: "PrepareTransaction",
    0x1A: "CommitTransaction",
    0x1B: "ForgetTransaction",
    0x1C: "OpenNonresidentAttribute",
    0x1D: "OpenAttributeTableDump",
    0x1E: "AttributeNamesDump",
    0x1F: "DirtyPageTableDump",
    0x20: "TransactionTableDump",
    0x21: "UpdateRecordDataRoot",
    0x22: "UpdateRecordDataAllocation",
    0x23: "UpdateRelativeDataIndex",
    0x24: "UpdateRelativeDataAllocation",
    0x25: "ZeroEndOfFileRecord"
}


def logfile_get_operation_name(op_code):
    """Get human-readable operation name."""
    return LOGFILE_OPERATION_CODES.get(op_code, f"Unknown_0x{op_code:02X}")


def logfile_filetime_to_datetime(filetime_bytes):
    """Convert FILETIME (8-byte little-endian) to datetime string."""
    if not filetime_bytes or len(filetime_bytes) != 8:
        return None
    
    try:
        filetime = struct.unpack('<Q', filetime_bytes)[0]
        EPOCH_DIFF = 116444736000000000
        
        if filetime == 0:
            return None
        
        microseconds = (filetime - EPOCH_DIFF) // 10
        dt = datetime(1970, 1, 1) + timedelta(microseconds=microseconds)
        return dt.strftime("%Y-%m-%d %H:%M:%S.%f")
    
    except (struct.error, ValueError, OSError, OverflowError):
        return None


def logfile_extract_timestamps_from_buffer(data_buffer, attribute_offset):
    """Extract timestamps from redo/undo data buffer based on attribute offset."""
    timestamps = {
        "C": None,
        "M": None,
        "E": None,
        "A": None
    }
    
    if not data_buffer or len(data_buffer) < 8:
        return timestamps
    
    try:
        if attribute_offset == 0x18:
            if len(data_buffer) >= 32:
                timestamps["C"] = logfile_filetime_to_datetime(data_buffer[0:8])
                timestamps["M"] = logfile_filetime_to_datetime(data_buffer[8:16])
                timestamps["E"] = logfile_filetime_to_datetime(data_buffer[16:24])
                timestamps["A"] = logfile_filetime_to_datetime(data_buffer[24:32])
        
        elif attribute_offset == 0x20:
            if len(data_buffer) >= 24:
                timestamps["M"] = logfile_filetime_to_datetime(data_buffer[0:8])
                timestamps["E"] = logfile_filetime_to_datetime(data_buffer[8:16])
                timestamps["A"] = logfile_filetime_to_datetime(data_buffer[16:24])
        
        elif attribute_offset == 0x28:
            if len(data_buffer) >= 16:
                timestamps["E"] = logfile_filetime_to_datetime(data_buffer[0:8])
                timestamps["A"] = logfile_filetime_to_datetime(data_buffer[8:16])
        
        elif attribute_offset == 0x30:
            if len(data_buffer) >= 8:
                timestamps["A"] = logfile_filetime_to_datetime(data_buffer[0:8])
    
    except Exception:
        pass
    
    return timestamps


def logfile_is_timestamp_change_record(record):
    """Check if a LogFile record represents a timestamp change event."""
    try:
        if record.get_redo_operation() != 0x07:
            return False
        
        record_offset = record.get_record_offset()
        if record_offset != 0x38:
            return False
        
        attr_offset = record.get_attribute_offset()
        if attr_offset < 0x18 or attr_offset > 0x30:
            return False
        
        return True
    
    except Exception:
        return False

print("LogFile helper functions defined.")


LogFile helper functions defined.


In [28]:
# [Cell 8] LogFile Parser - Main Function

def parse_logfile(logfile_path, progress_interval=5000):
    """
    Parse the LogFile and extract all relevant transaction records.
    
    Args:
        logfile_path: Path to the $LogFile
        progress_interval: Print progress every N records
        
    Returns:
        pandas.DataFrame: Parsed LogFile data
    """
    records = []
    
    print(f"  Opening LogFile: {logfile_path.name}")
    
    with open(logfile_path, "rb") as logfile:
        parser = LogFileParser(logfile)
        parser.collect_lsns()
        
        record_count = 0
        timestamp_change_count = 0
        error_count = 0
        
        for record in parser.parse_ntfs_records():
            try:
                record_count += 1
                
                if record_count % progress_interval == 0:
                    print(f"    Processed {record_count:,} records (TS changes: {timestamp_change_count:,})...")
                
                lsn = record.lsn
                redo_op = record.get_redo_operation()
                undo_op = record.get_undo_operation()
                redo_op_name = logfile_get_operation_name(redo_op)
                undo_op_name = logfile_get_operation_name(undo_op)
                
                try:
                    record_offset = record.get_record_offset()
                except Exception:
                    record_offset = None
                
                try:
                    attribute_offset = record.get_attribute_offset()
                except Exception:
                    attribute_offset = None
                
                try:
                    target_vcn = record.get_target_vcn()
                except Exception:
                    target_vcn = None
                
                try:
                    mft_target_number = record.calculate_mft_target_number()
                except Exception:
                    mft_target_number = None
                
                try:
                    redo_data = record.get_redo_data()
                except Exception:
                    redo_data = None
                
                try:
                    undo_data = record.get_undo_data()
                except Exception:
                    undo_data = None
                
                is_timestamp_change = logfile_is_timestamp_change_record(record)
                
                undo_timestamps = {"C": None, "M": None, "E": None, "A": None}
                redo_timestamps = {"C": None, "M": None, "E": None, "A": None}
                
                if is_timestamp_change and attribute_offset is not None:
                    timestamp_change_count += 1
                    if undo_data:
                        undo_timestamps = logfile_extract_timestamps_from_buffer(undo_data, attribute_offset)
                    if redo_data:
                        redo_timestamps = logfile_extract_timestamps_from_buffer(redo_data, attribute_offset)
                
                parsed_record = {
                    "LSN": lsn,
                    "RedoOP": redo_op,
                    "UndoOP": undo_op,
                    "RedoOPName": redo_op_name,
                    "UndoOPName": undo_op_name,
                    "RecordOffset": record_offset,
                    "AttributeOffset": attribute_offset,
                    "TargetVCN": target_vcn,
                    "TargetFRN": mft_target_number,
                    "IsTimestampChange": is_timestamp_change,
                    "Undo_$SI-C": undo_timestamps["C"],
                    "Undo_$SI-M": undo_timestamps["M"],
                    "Undo_$SI-E": undo_timestamps["E"],
                    "Undo_$SI-A": undo_timestamps["A"],
                    "Redo_$SI-C": redo_timestamps["C"],
                    "Redo_$SI-M": redo_timestamps["M"],
                    "Redo_$SI-E": redo_timestamps["E"],
                    "Redo_$SI-A": redo_timestamps["A"]
                }
                
                records.append(parsed_record)
                
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f"    Warning: Error parsing record {record_count}: {str(e)[:80]}")
    
    print(f"    Total: {record_count:,} records, TS changes: {timestamp_change_count:,}, Errors: {error_count:,}")
    
    return pd.DataFrame(records)

print("LogFile parser function defined.")


LogFile parser function defined.


## Batch Processing Function

The following cell defines the main batch processing function that orchestrates parsing all three artifacts.


In [29]:
# [Cell 9] Batch Processing Function

def process_dataset(dataset_name, category, input_base_dir, output_base_dir):
    """
    Process a single dataset: parse MFT, LogFile, and UsnJrnl.
    
    Args:
        dataset_name: Name of the dataset (e.g., "01-PE", "01-APT17")
        category: Dataset category ("PE", "APT", or "LoneWolf")
        input_base_dir: Base directory for raw data
        output_base_dir: Base directory for output
        
    Returns:
        dict: Statistics for the processed dataset
    """
    print("\n" + "=" * 70)
    print(f"PROCESSING DATASET: {dataset_name} ({category})")
    print("=" * 70)
    
    # Define paths
    input_dir = input_base_dir / category / dataset_name
    output_dir = output_base_dir / category / dataset_name
    
    # Create output directory
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Define file paths
    mft_input = input_dir / "$MFT"
    logfile_input = input_dir / "$LogFile"
    usnjrnl_input = input_dir / "$UsnJrnl_$J.bin"
    
    mft_output = output_dir / f"{dataset_name}-MFT.csv"
    logfile_output = output_dir / f"{dataset_name}-LogFile.csv"
    usnjrnl_output = output_dir / f"{dataset_name}-UsnJrnl.csv"
    
    # Verify input files exist
    missing_files = []
    for filepath in [mft_input, logfile_input, usnjrnl_input]:
        if not filepath.exists():
            missing_files.append(filepath.name)
    
    if missing_files:
        print(f"ERROR: Missing input files: {', '.join(missing_files)}")
        return None
    
    stats = {
        "dataset": dataset_name,
        "category": category,
        "mft_records": 0,
        "logfile_records": 0,
        "usnjrnl_records": 0,
        "timestamp_changes": 0,
        "basic_info_changes": 0,
        "mft_input_size_mb": 0,
        "mft_output_size_mb": 0,
        "logfile_input_size_mb": 0,
        "logfile_output_size_mb": 0,
        "usnjrnl_input_size_mb": 0,
        "usnjrnl_output_size_mb": 0
    }
    
    # Parse MFT
    print("\n[1/3] Parsing MFT...")
    stats["mft_input_size_mb"] = mft_input.stat().st_size / (1024 * 1024)
    print(f"  Input file size: {stats['mft_input_size_mb']:.2f} MB")
    
    df_mft = parse_mft(mft_input)
    df_mft.to_csv(mft_output, index=False, encoding="utf-8")
    
    stats["mft_records"] = len(df_mft)
    stats["mft_output_size_mb"] = mft_output.stat().st_size / (1024 * 1024)
    
    print(f"  Saved: {mft_output.name} ({len(df_mft):,} records)")
    print(f"  Output file size: {stats['mft_output_size_mb']:.2f} MB")
    print(f"  Compression ratio: {stats['mft_input_size_mb'] / stats['mft_output_size_mb']:.2f}x")
    
    # Print MFT statistics
    print(f"  Active entries: {df_mft['IsActive'].sum():,}")
    print(f"  With $SI timestamps: {df_mft['$SI-C'].notna().sum():,}")
    print(f"  With $FN timestamps: {df_mft['$FN-C'].notna().sum():,}")
    
    # Parse LogFile
    print("\n[2/3] Parsing LogFile...")
    stats["logfile_input_size_mb"] = logfile_input.stat().st_size / (1024 * 1024)
    print(f"  Input file size: {stats['logfile_input_size_mb']:.2f} MB")
    
    df_logfile = parse_logfile(logfile_input)
    df_logfile.to_csv(logfile_output, index=False, encoding="utf-8")
    
    stats["logfile_records"] = len(df_logfile)
    stats["timestamp_changes"] = df_logfile["IsTimestampChange"].sum()
    stats["logfile_output_size_mb"] = logfile_output.stat().st_size / (1024 * 1024)
    
    print(f"  Saved: {logfile_output.name} ({len(df_logfile):,} records)")
    print(f"  Output file size: {stats['logfile_output_size_mb']:.2f} MB")
    print(f"  Compression ratio: {stats['logfile_input_size_mb'] / stats['logfile_output_size_mb']:.2f}x")
    
    # Print LogFile statistics
    print(f"  Timestamp changes: {stats['timestamp_changes']:,}")
    print(f"  UpdateResidentValue (0x07): {len(df_logfile[df_logfile['RedoOP'] == 0x07]):,}")
    
    # Parse UsnJrnl
    print("\n[3/3] Parsing UsnJrnl...")
    stats["usnjrnl_input_size_mb"] = usnjrnl_input.stat().st_size / (1024 * 1024)
    print(f"  Input file size: {stats['usnjrnl_input_size_mb']:.2f} MB")
    
    df_usnjrnl = parse_usnjrnl(usnjrnl_input)
    df_usnjrnl.to_csv(usnjrnl_output, index=False, encoding="utf-8")
    
    stats["usnjrnl_records"] = len(df_usnjrnl)
    stats["basic_info_changes"] = df_usnjrnl["HasBasicInfoChange"].sum()
    stats["usnjrnl_output_size_mb"] = usnjrnl_output.stat().st_size / (1024 * 1024)
    
    print(f"  Saved: {usnjrnl_output.name} ({len(df_usnjrnl):,} records)")
    print(f"  Output file size: {stats['usnjrnl_output_size_mb']:.2f} MB")
    print(f"  Compression ratio: {stats['usnjrnl_input_size_mb'] / stats['usnjrnl_output_size_mb']:.2f}x")
    
    # Print UsnJrnl statistics
    print(f"  BASIC_INFO_CHANGE: {stats['basic_info_changes']:,}")
    print(f"  CLOSE events: {df_usnjrnl['HasClose'].sum():,}")
    print(f"  FILE_CREATE events: {df_usnjrnl['HasFileCreate'].sum():,}")
    
    # Final summary
    total_input_mb = stats["mft_input_size_mb"] + stats["logfile_input_size_mb"] + stats["usnjrnl_input_size_mb"]
    total_output_mb = stats["mft_output_size_mb"] + stats["logfile_output_size_mb"] + stats["usnjrnl_output_size_mb"]
    
    print("\n" + "=" * 70)
    print(f"COMPLETED: {dataset_name}")
    print("=" * 70)
    print(f"Total input size:  {total_input_mb:.2f} MB")
    print(f"Total output size: {total_output_mb:.2f} MB")
    print(f"Overall compression: {total_input_mb / total_output_mb:.2f}x")
    print("=" * 70)
    
    return stats

print("Batch processing function defined.")


Batch processing function defined.


## Dataset Processing

The following cells process individual datasets. Execute them incrementally as needed.

### Processing Order:
1. PE datasets (01-PE to 12-PE)
2. APT datasets (01-APT17 to 14-Winnti43b)
3. LoneWolf dataset

Each cell can be run independently.

In [30]:
# [Cell 10] Process PE Dataset: 01-PE

dataset_name = "01-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)


PROCESSING DATASET: 01-PE (PE)

[1/3] Parsing MFT...
  Input file size: 515.25 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Total: 519,113 records, Errors: 0, Parsed: 519,113
  Saved: 01-PE-MFT.csv (519,113 records)
  Output file size: 223.24 MB
  Compression ratio: 2.31x
  Active entries: 517,015
  With $SI timestamps: 519,113
  With $FN timestamps: 519,109

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 126)...
    Processed 10,000 records (TS changes: 226)...
    Processed 15,000 records (TS changes: 356)...
    Processed 20,000 records (TS changes: 484)...
    Processed 25,000 records (TS changes:

In [31]:
# [Cell 11] Process PE Dataset: 02-PE

dataset_name = "02-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)


PROCESSING DATASET: 02-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,470 records, Errors: 0, Parsed: 685,470
  Saved: 02-PE-MFT.csv (685,470 records)
  Output file size: 292.79 MB
  Compression ratio: 2.34x
  Active entries: 378,488
  With $SI timestamps: 685,470
  With $FN timestamps: 685,449

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 293)...
    Processed 10,000 records (TS changes: 874)...
    Processed 15,000 records (TS changes: 

In [32]:
# [Cell 12] Process PE Dataset: 03-PE

dataset_name = "03-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)


PROCESSING DATASET: 03-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,469 records, Errors: 0, Parsed: 685,469
  Saved: 03-PE-MFT.csv (685,469 records)
  Output file size: 292.79 MB
  Compression ratio: 2.34x
  Active entries: 378,486
  With $SI timestamps: 685,469
  With $FN timestamps: 685,448

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 311)...
    Processed 10,000 records (TS changes: 531)...
    Processed 15,000 records (TS changes: 

In [33]:
# [Cell 13] Process PE Dataset: 04-PE

dataset_name = "04-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 04-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,543 records, Errors: 0, Parsed: 685,543
  Saved: 04-PE-MFT.csv (685,543 records)
  Output file size: 292.74 MB
  Compression ratio: 2.34x
  Active entries: 379,439
  With $SI timestamps: 685,543
  With $FN timestamps: 685,522

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 205)...
    Processed 10,000 records (TS changes: 388)...
    Processed 15,000 records (TS changes: 

In [34]:
# [Cell 14] Process PE Dataset: 05-PE

dataset_name = "05-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 05-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,545 records, Errors: 0, Parsed: 685,545
  Saved: 05-PE-MFT.csv (685,545 records)
  Output file size: 292.74 MB
  Compression ratio: 2.34x
  Active entries: 379,429
  With $SI timestamps: 685,545
  With $FN timestamps: 685,524

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 169)...
    Processed 10,000 records (TS changes: 374)...
    Processed 15,000 records (TS changes: 

In [35]:
# [Cell 15] Process PE Dataset: 06-PE

dataset_name = "06-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 06-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,544 records, Errors: 0, Parsed: 685,544
  Saved: 06-PE-MFT.csv (685,544 records)
  Output file size: 292.73 MB
  Compression ratio: 2.34x
  Active entries: 379,425
  With $SI timestamps: 685,544
  With $FN timestamps: 685,523

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 200)...
    Processed 10,000 records (TS changes: 452)...
    Processed 15,000 records (TS changes: 

In [37]:
# [Cell 16] Process PE Dataset: 07-PE

dataset_name = "07-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 07-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,483 records, Errors: 0, Parsed: 685,483
  Saved: 07-PE-MFT.csv (685,483 records)
  Output file size: 292.78 MB
  Compression ratio: 2.34x
  Active entries: 378,307
  With $SI timestamps: 685,483
  With $FN timestamps: 685,462

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 311)...
    Processed 10,000 records (TS changes: 531)...
    Processed 15,000 records (TS changes: 

In [38]:
# [Cell 17] Process PE Dataset: 08-PE

dataset_name = "08-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 08-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,483 records, Errors: 0, Parsed: 685,483
  Saved: 08-PE-MFT.csv (685,483 records)
  Output file size: 292.78 MB
  Compression ratio: 2.34x
  Active entries: 378,310
  With $SI timestamps: 685,483
  With $FN timestamps: 685,462

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 263)...
    Processed 10,000 records (TS changes: 476)...
    Processed 15,000 records (TS changes: 

In [39]:
# [Cell 18] Process PE Dataset: 09-PE

dataset_name = "09-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 09-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,495 records, Errors: 0, Parsed: 685,495
  Saved: 09-PE-MFT.csv (685,495 records)
  Output file size: 292.79 MB
  Compression ratio: 2.34x
  Active entries: 378,275
  With $SI timestamps: 685,495
  With $FN timestamps: 685,474

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 217)...
    Processed 10,000 records (TS changes: 429)...
    Processed 15,000 records (TS changes: 

In [40]:
# [Cell 19] Process PE Dataset: 10-PE

dataset_name = "10-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 10-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,497 records, Errors: 0, Parsed: 685,497
  Saved: 10-PE-MFT.csv (685,497 records)
  Output file size: 292.78 MB
  Compression ratio: 2.34x
  Active entries: 378,393
  With $SI timestamps: 685,497
  With $FN timestamps: 685,476

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 263)...
    Processed 10,000 records (TS changes: 480)...
    Processed 15,000 records (TS changes: 

In [41]:
# [Cell 20] Process PE Dataset: 11-PE

dataset_name = "11-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 11-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,544 records, Errors: 0, Parsed: 685,544
  Saved: 11-PE-MFT.csv (685,544 records)
  Output file size: 292.73 MB
  Compression ratio: 2.34x
  Active entries: 379,425
  With $SI timestamps: 685,544
  With $FN timestamps: 685,523

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 194)...
    Processed 10,000 records (TS changes: 469)...
    Processed 15,000 records (TS changes: 

In [42]:
# [Cell 21] Process PE Dataset: 12-PE

dataset_name = "12-PE"
stats = process_dataset(dataset_name, "PE", RAW_DATA_DIR, OUTPUT_BASE_DIR)



PROCESSING DATASET: 12-PE (PE)

[1/3] Parsing MFT...
  Input file size: 684.75 MB
  Opening MFT file: $MFT
    Processed 50,000 records...
    Processed 100,000 records...
    Processed 150,000 records...
    Processed 200,000 records...
    Processed 250,000 records...
    Processed 300,000 records...
    Processed 350,000 records...
    Processed 400,000 records...
    Processed 450,000 records...
    Processed 500,000 records...
    Processed 550,000 records...
    Processed 600,000 records...
    Processed 650,000 records...
    Total: 685,551 records, Errors: 0, Parsed: 685,551
  Saved: 12-PE-MFT.csv (685,551 records)
  Output file size: 292.74 MB
  Compression ratio: 2.34x
  Active entries: 379,428
  With $SI timestamps: 685,551
  With $FN timestamps: 685,530

[2/3] Parsing LogFile...
  Input file size: 64.00 MB
  Opening LogFile: $LogFile
    Processed 5,000 records (TS changes: 170)...
    Processed 10,000 records (TS changes: 360)...
    Processed 15,000 records (TS changes: 

In [ ]:
# [Cell 22] Process APT Dataset: 01-APT17

dataset_name = "01-APT17"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 23] Process APT Dataset: 02-APT19

dataset_name = "02-APT19"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 24] Process APT Dataset: 03-APT21

dataset_name = "03-APT21"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 25] Process APT Dataset: 04-APT28

dataset_name = "04-APT28"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 26] Process APT Dataset: 05-APT29

dataset_name = "05-APT29"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 27] Process APT Dataset: 06-APT30

dataset_name = "06-APT30"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 28] Process APT Dataset: 07-APT37

dataset_name = "07-APT37"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 29] Process APT Dataset: 08-APT38

dataset_name = "08-APT38"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 30] Process APT Dataset: 09-APT40

dataset_name = "09-APT40"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 31] Process APT Dataset: 10-DarkHotel663

dataset_name = "10-DarkHotel663"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 32] Process APT Dataset: 11-DarkHotelbbd

dataset_name = "11-DarkHotelbbd"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 33] Process APT Dataset: 12-Kimsuky

dataset_name = "12-Kimsuky"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 34] Process APT Dataset: 13-Winnti731

dataset_name = "13-Winnti731"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 35] Process APT Dataset: 14-Winnti43b

dataset_name = "14-Winnti43b"
stats = process_dataset(dataset_name, "APT", RAW_DATA_DIR, OUTPUT_BASE_DIR)


In [ ]:
# [Cell 36] Process LoneWolf Dataset

dataset_name = "LoneWolf"
stats = process_dataset(dataset_name, "LoneWolf", RAW_DATA_DIR, OUTPUT_BASE_DIR)


## Summary

All datasets have been processed. The parsed CSV files are now ready for Phase 2.

### Next Steps:
1. **Phase 2**: Data Preprocessing and Event Grouping
2. **Phase 3**: Feature Engineering and Labeling
3. **Phase 4**: Model Training and Evaluation